# Consensus heatmap — Aβ42 variants

Tools: TANGO, PASTA, CrossBeta, AmyPred-FRL.

In [ ]:
import pandas as pd
import numpy as np
import io
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.cluster.hierarchy import linkage, leaves_list

DATA_DIR = "../predictions/"

BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
WT_COL = "#4ff7c0"

# Clinically relevant mutations — highlight separately
CLINICAL = {
    "E22G_Arctic_Abeta42",
    "E22Q_Dutch_Abeta42",
    "D23N_Iowa_Abeta42",
    "A2T_Icelandic_Abeta42",
}

In [ ]:
# 1. Load data → Series {name: delta}


def parse_name(name):
    """Return a short readable variant name."""
    s = name.replace("_Abeta42", "").replace("_Abeta_42", "")
    return s


def load_tango(path):
    df = pd.read_csv(path, sep="\t")
    wt = df.loc[df["Sequence"] == "Wildtype_Abeta_42", "Aggregation"].iloc[0]
    out = {}
    for _, row in df.iterrows():
        if "Wildtype" in row["Sequence"]:
            continue
        out[row["Sequence"]] = row["Aggregation"] - wt
    return pd.Series(out, name="TANGO\nΔ Aggregation")


def load_pasta(path):
    with open(path, encoding="utf-8-sig") as f:
        df = pd.read_csv(io.StringIO(f.read().replace(",", ".")), sep=";")
    wt = df.loc[df["Protein name"].str.contains("Wildtype"), "Best Energy"].iloc[0]
    out = {}
    for _, row in df.iterrows():
        if "Wildtype" in row["Protein name"]:
            continue
        out[row["Protein name"]] = row["Best Energy"] - wt
    return pd.Series(out, name="PASTA\nΔ Energy")


def load_amypred(path):
    lines = open(path, encoding="utf-8-sig").readlines()
    data = {}
    wt_val = None
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(";")
        name, prob = parts[0], float(parts[2])
        if "Wildtype" in name:
            wt_val = prob
        else:
            data[name] = prob
    if wt_val is not None:
        data = {k: v - wt_val for k, v in data.items()}
    return pd.Series(data, name="AmyPred-FRL\nΔ Probability")


def load_crossbeta(path):
    lines = open(path).readlines()
    data = {}
    wt_val = None
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(";")
        name, avg = parts[0], float(parts[2])
        if "Wildtype" in name:
            wt_val = avg
        else:
            data[name] = avg
    if wt_val is not None:
        data = {k: v - wt_val for k, v in data.items()}
    return pd.Series(data, name="CrossBeta\nΔ Avg Score")

In [ ]:
tango = load_tango(DATA_DIR + "tango/tango_input_aggregation.txt")
pasta = load_pasta(DATA_DIR + "pasta/pasta.csv")
amypred = load_amypred(DATA_DIR + "amypred/amypred-frl.csv")
crossbeta = load_crossbeta(DATA_DIR + "cross-beta/cross-beta_result.csv")

# Combine into a single DataFrame
raw = pd.DataFrame([tango, pasta, amypred, crossbeta]).T
raw.index.name = "variant"

# Keep only variants present in all tools
raw = raw.dropna()
print(f"Variants in matrix: {len(raw)}")

In [ ]:
# 2. Normalize to z-score per tool
zscore = (raw - raw.mean()) / raw.std()

# 3. Hierarchical clustering over variants
Z = linkage(zscore.values, method="ward", metric="euclidean")
order = leaves_list(Z)
zscore_ordered = zscore.iloc[order]
raw_ordered = raw.iloc[order]

# 4. Short names for Y axis
short_names = [parse_name(n) for n in zscore_ordered.index]
is_clinical = [n in CLINICAL for n in zscore_ordered.index]

# 5. Draw figure
n_var = len(zscore_ordered)
n_tool = len(zscore_ordered.columns)

fig = plt.figure(figsize=(16, 22), facecolor=BG)

# Grid: dendrogram on the left, heatmap in the middle, consensus column on the right
gs = GridSpec(
    1,
    3,
    figure=fig,
    width_ratios=[4, 0.35, 1.1],
    wspace=0.04,
)

ax_heat = fig.add_subplot(gs[0])
ax_cons = fig.add_subplot(gs[1])
ax_raw = fig.add_subplot(gs[2])

for ax in [ax_heat, ax_cons, ax_raw]:
    ax.set_facecolor(BG)

# 5b. Main heatmap (z-score)
vlim = np.percentile(np.abs(zscore.values), 98)
norm_z = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
cmap_z = matplotlib.colormaps["RdBu_r"]

mat = zscore_ordered.values  # (n_var, n_tool)

ax_heat.imshow(
    mat,
    aspect="auto",
    cmap=cmap_z,
    norm=norm_z,
    interpolation="nearest",
    origin="upper",
)

# Grid lines
for x in np.arange(-0.5, n_tool, 1):
    ax_heat.axvline(x, color=BG, lw=1.2, zorder=3)

# Numbers in cells for outliers
for i in range(n_var):
    for j in range(n_tool):
        v = mat[i, j]
        if abs(v) > vlim * 0.65:
            tc = "white" if abs(v) > vlim * 0.85 else TEXT
            ax_heat.text(
                j,
                i,
                f"{v:+.1f}",
                ha="center",
                va="center",
                fontsize=5.5,
                fontfamily="monospace",
                color=tc,
                fontweight="bold",
            )

# X labels
ax_heat.set_xticks(range(n_tool))
ax_heat.set_xticklabels(
    zscore_ordered.columns,
    fontsize=8.5,
    fontfamily="monospace",
    color=TEXT,
    ha="center",
    va="bottom",
)
ax_heat.tick_params(
    axis="x", length=0, pad=6, top=True, bottom=False, labeltop=True, labelbottom=False
)

# Y labels — variant names
ax_heat.set_yticks(range(n_var))
ax_heat.set_yticklabels(
    short_names,
    fontsize=6.8,
    fontfamily="monospace",
    color=TEXT,
)
ax_heat.tick_params(axis="y", length=0, pad=4)
for tick, clin in zip(ax_heat.yaxis.get_ticklabels(), is_clinical):
    tick.set_color("#fbbf24" if clin else TEXT)
ax_heat.set_ylim(n_var - 0.5, -0.5)

# Horizontal bands
for i in range(n_var):
    if i % 2 == 0:
        ax_heat.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.025, zorder=0)

for sp in ax_heat.spines.values():
    sp.set_visible(False)

ax_heat.set_title(
    "z-score per tool", color=MUTED, fontsize=8, fontfamily="monospace", pad=28
)

# Colorbar for z-score
sm_z = plt.cm.ScalarMappable(cmap=cmap_z, norm=norm_z)
sm_z.set_array([])
cbar = plt.colorbar(
    sm_z,
    ax=ax_heat,
    shrink=0.25,
    pad=0.01,
    aspect=20,
    orientation="vertical",
    anchor=(0, 0.02),
    panchor=(0, 0.02),
)
cbar.set_label("z-score", color=MUTED, fontsize=7, fontfamily="monospace")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=6.5)
cbar.outline.set_edgecolor("#1e2540")

# 5c. Consensus column: mean z-score
consensus = zscore_ordered.mean(axis=1).values

norm_c = mcolors.TwoSlopeNorm(
    vmin=min(consensus.min(), -0.1),
    vcenter=0,
    vmax=max(consensus.max(), 0.1),
)
cmap_c = matplotlib.colormaps["RdBu_r"]

for i, v in enumerate(consensus):
    color = cmap_c(norm_c(v))
    ax_cons.barh(i, 1, left=0, color=color, height=0.85)
    if abs(v) > 0.4:
        tc = "white" if abs(v) > 0.8 else TEXT
        ax_cons.text(
            0.5,
            i,
            f"{v:+.2f}",
            ha="center",
            va="center",
            fontsize=5.8,
            fontfamily="monospace",
            color=tc,
            fontweight="bold",
        )

ax_cons.set_xlim(0, 1)
ax_cons.set_ylim(n_var - 0.5, -0.5)
ax_cons.set_xticks([])
ax_cons.set_yticks([])
ax_cons.set_title(
    "mean\nz", color=MUTED, fontsize=7.5, fontfamily="monospace", pad=6, linespacing=1.3
)

for sp in ax_cons.spines.values():
    sp.set_visible(False)
ax_cons.axvline(0.5, color=BG, lw=0.5)

# Colorbar for consensus
sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
sm_c.set_array([])
cbar2 = plt.colorbar(
    sm_c,
    ax=ax_cons,
    shrink=0.25,
    pad=0.15,
    aspect=20,
    orientation="vertical",
    anchor=(0, 0.02),
    panchor=(0, 0.02),
)
cbar2.set_label(
    "consensus\nz-score",
    color=MUTED,
    fontsize=6.5,
    fontfamily="monospace",
    linespacing=1.3,
)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=6)
cbar2.outline.set_edgecolor("#1e2540")

# 5d. Raw values (dot plot)
# Normalize each tool to [0,1] for display
tool_colors = ["#ef4444", "#3b82f6", "#a78bfa", "#4ff7c0"]
tool_cols = raw_ordered.columns.tolist()

for j, (col, tc) in enumerate(zip(tool_cols, tool_colors)):
    vals = raw_ordered[col].values
    vmin_, vmax_ = vals.min(), vals.max()
    norm_01 = (vals - vmin_) / (vmax_ - vmin_ + 1e-12)

    for i, v01 in enumerate(norm_01):
        raw_v = vals[i]
        ax_raw.scatter(
            j * 1.1, i, s=18, c=[tc], alpha=0.5 + 0.5 * v01, zorder=3, linewidths=0
        )

ax_raw.set_xlim(-0.6, n_tool * 1.1 - 0.5)
ax_raw.set_ylim(n_var - 0.5, -0.5)
ax_raw.set_xticks([j * 1.1 for j in range(n_tool)])
ax_raw.set_xticklabels(
    [c.split("\n")[0] for c in tool_cols],
    fontsize=7,
    fontfamily="monospace",
    color=MUTED,
    ha="center",
    va="bottom",
    rotation=0,
)
ax_raw.tick_params(
    axis="x", length=0, pad=6, top=True, bottom=False, labeltop=True, labelbottom=False
)
ax_raw.set_yticks([])
ax_raw.set_title(
    "raw Δ\n(scaled dot)",
    color=MUTED,
    fontsize=7.5,
    fontfamily="monospace",
    pad=28,
    linespacing=1.3,
)

for sp in ax_raw.spines.values():
    sp.set_visible(False)
for i in range(n_var):
    if i % 2 == 0:
        ax_raw.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.025, zorder=0)

# Main title + legend
fig.suptitle(
    "Consensus Heatmap  ·  Aβ42 variants  ·  4 amyloid prediction tools\n"
    "z-score per tool  ·  Ward clustering  ·  "
    "red = higher than WT  ·  blue = lower than WT",
    color="white",
    fontsize=11,
    fontfamily="monospace",
    fontweight="bold",
    y=0.995,
)

clinical_patch = mpatches.Patch(
    facecolor="#fbbf24", label="Clinically relevant mutations"
)
fig.legend(
    handles=[clinical_patch],
    loc="lower left",
    bbox_to_anchor=(0.01, 0.001),
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    facecolor=BG,
    fontsize=8,
    labelcolor=TEXT,
)

plt.show()
plt.savefig(
    "consensus_heatmap.png",
    dpi=180,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to consensus_heatmap.png")
plt.close()